In [17]:
import torch
print("File    :", torch.__file__)
print("Version :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))
    print("VRAM    :", torch.cuda.get_device_properties(0))

File    : d:\End-to-end-ML\Customer-Risk-Escalation-Engine\venv\Lib\site-packages\torch\__init__.py
Version : 2.5.1+cu121
CUDA    : True
GPU     : NVIDIA GeForce RTX 3050 Laptop GPU
VRAM    : _CudaDeviceProperties(name='NVIDIA GeForce RTX 3050 Laptop GPU', major=8, minor=6, total_memory=4095MB, multi_processor_count=16, uuid=a6e1043e-91a3-e035-f030-fec0eab1f833, L2_cache_size=1MB)


In [2]:
import os
import numpy as np
import pandas as pd
import torch
from transformers import DistilBertTokenizer, DistilBertModel
from torch.utils.data import Dataset, DataLoader
import mlflow
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

os.chdir(r'D:\End-to-end-ML\Customer-Risk-Escalation-Engine')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

d:\End-to-end-ML\Customer-Risk-Escalation-Engine\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
text_train = pd.read_csv('data/processed_data/text_train.csv')['combined_text']
text_test  = pd.read_csv('data/processed_data/text_test.csv')['combined_text']

text_train = text_train.fillna('no text available').reset_index(drop=True)
text_test  = text_test.fillna('no text available').reset_index(drop=True)


In [4]:
print(f"text_train : {text_train.shape}") # type: ignore
print(f"text_test  : {text_test.shape}")  # type: ignore

text_train : (80000,)
text_test  : (20000,)


In [ ]:
MODEL_NAME = 'distilbert-base-uncased'

print("Loading tokenizer...")
tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)

print("Loading model...")
model = DistilBertModel.from_pretrained(MODEL_NAME)
model = model.to(device)
model.eval()  # Inference mode — no training happening

print(f"\nModel loaded on : {device}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading tokenizer...
Loading model...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2784.60it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Model loaded on : cuda ✅
Model parameters: 66,362,880


In [ ]:
class TicketTextDataset(Dataset):
    """
    Converts raw text strings into tokenized input
    that DistilBERT can understand.
    """
    def __init__(self, texts, tokenizer, max_length=128):
        self.texts     = texts.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])

        encoding = self.tokenizer(
            text,
            max_length      = self.max_length,
            padding         = 'max_length',
            truncation      = True,
            return_tensors  = 'pt'
        )

        return {
            'input_ids'      : encoding['input_ids'].squeeze(),
            'attention_mask' : encoding['attention_mask'].squeeze()
        }

print("Dataset class defined")

Dataset class defined


In [7]:
def extract_embeddings(texts, tokenizer, model, device,
                       batch_size=32, max_length=128):
    """
    Extracts 768-dim CLS embeddings from DistilBERT
    for a list of texts. Runs in batches to fit in VRAM.
    """
    dataset    = TicketTextDataset(texts, tokenizer, max_length)
    dataloader = DataLoader(dataset, batch_size=batch_size, 
                            shuffle=False, num_workers=0)

    all_embeddings = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Extracting embeddings"):

            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            outputs = model(
                input_ids      = input_ids,
                attention_mask = attention_mask
            )

            # CLS token = index 0 of last hidden state
            # Shape: (batch_size, 768)
            cls_embeddings = outputs.last_hidden_state[:, 0, :]

            # Move back to CPU and convert to numpy
            all_embeddings.append(cls_embeddings.cpu().numpy())

    return np.vstack(all_embeddings)


In [8]:
print("Extracting TRAIN embeddings...")
print(f"Total texts : {len(text_train):,}")
print(f"Batch size  : 32")
print(f"Device      : {device}\n")

train_embeddings = extract_embeddings(
    text_train, tokenizer, model, device,
    batch_size=32,
    max_length=128
)

print(f"\nTrain embeddings shape : {train_embeddings.shape}")


Extracting TRAIN embeddings...
Total texts : 80,000
Batch size  : 32
Device      : cuda



Extracting embeddings:   0%|          | 0/2500 [00:00<?, ?it/s]

Extracting embeddings: 100%|██████████| 2500/2500 [08:56<00:00,  4.66it/s]


Train embeddings shape : (80000, 768)


In [9]:
print("Extracting TEST embeddings...")
print(f"Total texts : {len(text_test):,}\n")

test_embeddings = extract_embeddings(
    text_test, tokenizer, model, device,
    batch_size=32,
    max_length=128
)

print(f"\nTest embeddings shape  : {test_embeddings.shape}")

Extracting TEST embeddings...
Total texts : 20,000



Extracting embeddings: 100%|██████████| 625/625 [02:11<00:00,  4.74it/s]


Test embeddings shape  : (20000, 768)


In [10]:
np.save('data/processed_data/train_embeddings.npy', train_embeddings)
np.save('data/processed_data/test_embeddings.npy',  test_embeddings)

print("Embeddings saved")

Embeddings saved


In [11]:
#MLflow logging
mlflow.set_tracking_uri(
    'file:///D:/End-to-end-ML/Customer-Risk-Escalation-Engine/mlruns'
)
mlflow.set_experiment("customer_escalation_nlp")

with mlflow.start_run(run_name="DistilBERT_Embeddings"):
    mlflow.log_param("model_name",    "distilbert-base-uncased")
    mlflow.log_param("max_length",    128)
    mlflow.log_param("batch_size",    32)
    mlflow.log_param("train_samples", len(text_train))
    mlflow.log_param("test_samples",  len(text_test))
    mlflow.log_param("embedding_dim", 768)
    mlflow.log_param("device",        str(device))

    print("MLflow run logged ✅")

Traceback (most recent call last):
  File "d:\End-to-end-ML\Customer-Risk-Escalation-Engine\venv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 383, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\End-to-end-ML\Customer-Risk-Escalation-Engine\venv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 481, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\End-to-end-ML\Customer-Risk-Escalation-Engine\venv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 1670, in _read_yaml
    return _read_helper(root, file_name, attempts_remaining=retries)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\End-to-end-ML\Customer-Risk-Escalation-Engine\venv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 1663, 

MLflow run logged ✅
